# Phase 5: SBERT Leaderboard Evaluation

Evaluates the latest SBERT run (or a selected model path) on unseen labeled data,
calibrates HATE/DISINFO thresholds on a train validation split, and appends metrics
to the leaderboard CSV.


In [ ]:
from pathlib import Path
from dataclasses import dataclass
from datetime import datetime, timezone
import csv
import json
import pickle
import re
import time

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path(r"D:/client-projects/sl-social-media-risk-analysis")
if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path.cwd().resolve()

MODEL_ROOT = Path("")  # optional override, e.g. PROJECT_ROOT / "training/artifacts/runs/run_x/model"
REPORT_DIR = Path("")  # optional override
UNSEEN_CSV = PROJECT_ROOT / "datasets/splits/current/unseen_labeled_rest.csv"
TRAIN_CSV = PROJECT_ROOT / "datasets/splits/current/train_labeled_331.csv"
LEADERBOARD_CSV = PROJECT_ROOT / "evaluation/reports/model_leaderboard.csv"
RUN_NAME = ""  # optional override
VAL_SIZE = 0.15
SEED = 42
BATCH_SIZE = 32
CHUNK_SIZE = 2000

def log(msg: str) -> None:
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"{now} | {msg}", flush=True)

def clean_text(text: str) -> str:
    if pd.isna(text):
        return ""
    value = str(text).replace("\u200d", "")
    value = " ".join(value.split()).strip()
    return value

def normalize_label(label: str) -> str:
    value = "" if pd.isna(label) else str(label)
    return re.sub(r"[,\s]+$", "", value.strip().upper())

def encode_in_chunks(model: SentenceTransformer, texts: list[str], *, batch_size: int, chunk_size: int) -> np.ndarray:
    total = len(texts)
    if total == 0:
        return np.empty((0, 0), dtype=np.float32)
    chunks: list[np.ndarray] = []
    start = time.perf_counter()
    log(f"Encoding start | rows={total} | chunk_size={chunk_size} | batch_size={batch_size}")
    for start_idx in range(0, total, chunk_size):
        end_idx = min(start_idx + chunk_size, total)
        log(f"Encoding chunk {start_idx + 1}-{end_idx}/{total}")
        emb = model.encode(
            texts[start_idx:end_idx],
            batch_size=batch_size,
            show_progress_bar=False,
            normalize_embeddings=True,
        )
        emb = np.asarray(emb, dtype=np.float32)
        chunks.append(emb)
        done = end_idx
        elapsed = max(time.perf_counter() - start, 1e-6)
        rate = done / elapsed
        eta = (total - done) / max(rate, 1e-6)
        log(f"Progress {done}/{total} | rate={rate:.1f} rows/s | eta={eta/60:.1f} min")
    arr = np.vstack(chunks)
    log(f"Encoding complete | shape={arr.shape}")
    return arr

def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray, label_order: list[str]) -> dict:
    report = classification_report(
        y_true,
        y_pred,
        labels=list(range(len(label_order))),
        target_names=label_order,
        output_dict=True,
        zero_division=0,
    )
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "report": report,
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=list(range(len(label_order)))).tolist(),
    }

def predict_with_thresholds(probs: np.ndarray, *, id_normal: int, id_hate: int, id_disinfo: int, t_hate: float, t_disinfo: float) -> np.ndarray:
    if probs.size == 0:
        return np.asarray([], dtype=int)
    preds = np.argmax(probs, axis=1).astype(int)
    for i in range(len(preds)):
        hate_p = float(probs[i, id_hate])
        disinfo_p = float(probs[i, id_disinfo])
        if hate_p >= t_hate or disinfo_p >= t_disinfo:
            preds[i] = id_hate if hate_p >= disinfo_p else id_disinfo
        else:
            preds[i] = id_normal
    return preds

@dataclass
class ThresholdResult:
    t_hate: float
    t_disinfo: float
    val_macro_f1: float
    val_accuracy: float
    val_min_class_f1: float

def search_thresholds(y_true: np.ndarray, probs: np.ndarray, *, id_normal: int, id_hate: int, id_disinfo: int, label_order: list[str]) -> ThresholdResult:
    best: ThresholdResult | None = None
    for t_hate in np.arange(0.30, 0.91, 0.05):
        for t_disinfo in np.arange(0.30, 0.91, 0.05):
            pred = predict_with_thresholds(
                probs,
                id_normal=id_normal,
                id_hate=id_hate,
                id_disinfo=id_disinfo,
                t_hate=float(t_hate),
                t_disinfo=float(t_disinfo),
            )
            m = compute_metrics(y_true, pred, label_order)
            rep = m["report"]
            class_f1 = [float(rep[label]["f1-score"]) for label in label_order]
            candidate = ThresholdResult(
                t_hate=float(t_hate),
                t_disinfo=float(t_disinfo),
                val_macro_f1=float(m["macro_f1"]),
                val_accuracy=float(m["accuracy"]),
                val_min_class_f1=min(class_f1),
            )
            if best is None:
                best = candidate
                continue
            if candidate.val_macro_f1 > best.val_macro_f1:
                best = candidate
                continue
            if candidate.val_macro_f1 == best.val_macro_f1 and candidate.val_min_class_f1 > best.val_min_class_f1:
                best = candidate
                continue
            if (
                candidate.val_macro_f1 == best.val_macro_f1
                and candidate.val_min_class_f1 == best.val_min_class_f1
                and candidate.val_accuracy > best.val_accuracy
            ):
                best = candidate
    assert best is not None
    return best

def append_leaderboard(path: Path, row: dict[str, str]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fieldnames = list(row.keys())
    exists = path.exists()
    with path.open("a", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        if not exists:
            writer.writeheader()
        writer.writerow(row)

def resolve_run_paths(project_root: Path, model_root_override: Path | None, report_dir_override: Path | None) -> tuple[Path, Path]:
    if model_root_override is not None and str(model_root_override).strip() != ".":
        model_root = model_root_override
        report_dir = report_dir_override if report_dir_override is not None and str(report_dir_override).strip() != "." else project_root / "training/artifacts/reports_phase5_sbert"
        return model_root, report_dir
    runs_root = project_root / "training/artifacts/runs"
    latest_file = runs_root / "latest_run.txt"
    if latest_file.exists():
        run_root = Path(latest_file.read_text(encoding="utf-8").strip())
        if not run_root.is_absolute():
            run_root = project_root / run_root
        model_root = run_root / "model"
        report_dir = report_dir_override if report_dir_override is not None and str(report_dir_override).strip() != "." else run_root / "reports"
        return model_root, report_dir
    model_root = project_root / "training/artifacts/models_phase5_sbert"
    report_dir = report_dir_override if report_dir_override is not None and str(report_dir_override).strip() != "." else project_root / "training/artifacts/reports_phase5_sbert"
    return model_root, report_dir

def load_eval_df(path: Path, label_order: list[str], label2id: dict[str, int]) -> pd.DataFrame:
    df = pd.read_csv(path)
    text_col = "clean_text" if "clean_text" in df.columns else "text"
    if text_col not in df.columns:
        raise ValueError(f"Expected text column in {path}")
    if "annotator_label" not in df.columns:
        raise ValueError(f"Expected annotator_label column in {path}")
    if "source" not in df.columns:
        df["source"] = "unknown"
    out = df.copy()
    out["text"] = out[text_col].apply(clean_text)
    out["label"] = out["annotator_label"].apply(normalize_label)
    out = out[(out["text"].str.len() > 0) & (out["label"].isin(label_order))].copy()
    out["label_id"] = out["label"].map(label2id).astype(int)
    return out

model_override = MODEL_ROOT if str(MODEL_ROOT).strip() else None
report_override = REPORT_DIR if str(REPORT_DIR).strip() else None
MODEL_ROOT_RESOLVED, REPORT_DIR_RESOLVED = resolve_run_paths(PROJECT_ROOT, model_override, report_override)
REPORT_DIR_RESOLVED.mkdir(parents=True, exist_ok=True)
MODEL_ROOT_RESOLVED, REPORT_DIR_RESOLVED


In [ ]:
meta_path = MODEL_ROOT_RESOLVED / "meta.json"
sbert_dir = MODEL_ROOT_RESOLVED / "sbert_model"
clf_path = MODEL_ROOT_RESOLVED / "embedding_classifier.pkl"
if not meta_path.exists() or not sbert_dir.exists() or not clf_path.exists():
    raise FileNotFoundError(f"Missing model artifacts under: {MODEL_ROOT_RESOLVED}")

meta = json.loads(meta_path.read_text(encoding="utf-8"))
label2id = {str(k): int(v) for k, v in meta["label2id"].items()}
id2label = {int(k): str(v) for k, v in meta["id2label"].items()}
label_order = [id2label[i] for i in sorted(id2label.keys())]
required = {"NORMAL", "HATE", "DISINFO"}
if not required.issubset(set(label_order)):
    raise ValueError(f"Model label space must include {required}. Got: {label_order}")

id_normal = label2id["NORMAL"]
id_hate = label2id["HATE"]
id_disinfo = label2id["DISINFO"]

log("Loading model artifacts...")
model = SentenceTransformer(str(sbert_dir))
with clf_path.open("rb") as f:
    clf = pickle.load(f)
if not hasattr(clf, "predict_proba"):
    raise ValueError("Classifier must support predict_proba for threshold calibration.")

log("Preparing calibration split from train CSV...")
train_df = load_eval_df(TRAIN_CSV, label_order, label2id)
_, val_df = train_test_split(
    train_df[["text", "label_id", "source"]],
    test_size=VAL_SIZE,
    random_state=SEED,
    stratify=train_df["label_id"],
)
log(f"Calibration rows: {len(val_df)}")

log("Encoding calibration split...")
x_val = encode_in_chunks(model, val_df["text"].tolist(), batch_size=BATCH_SIZE, chunk_size=CHUNK_SIZE)
y_val = val_df["label_id"].to_numpy()
p_val = clf.predict_proba(x_val)
log("Searching class thresholds on calibration split...")
best = search_thresholds(
    y_val,
    p_val,
    id_normal=id_normal,
    id_hate=id_hate,
    id_disinfo=id_disinfo,
    label_order=label_order,
)
log(f"Best thresholds | HATE={best.t_hate:.2f} DISINFO={best.t_disinfo:.2f} val_macro_f1={best.val_macro_f1:.4f}")

log("Loading unseen dataset...")
unseen_df = load_eval_df(UNSEEN_CSV, label_order, label2id)
log(f"Unseen rows: {len(unseen_df)}")
log("Encoding unseen dataset...")
x_unseen = encode_in_chunks(model, unseen_df["text"].tolist(), batch_size=BATCH_SIZE, chunk_size=CHUNK_SIZE)
y_true = unseen_df["label_id"].to_numpy()
p_unseen = clf.predict_proba(x_unseen)
y_argmax = np.argmax(p_unseen, axis=1).astype(int)
y_calib = predict_with_thresholds(
    p_unseen,
    id_normal=id_normal,
    id_hate=id_hate,
    id_disinfo=id_disinfo,
    t_hate=best.t_hate,
    t_disinfo=best.t_disinfo,
)

m_argmax = compute_metrics(y_true, y_argmax, label_order)
m_calib = compute_metrics(y_true, y_calib, label_order)

pred_df = unseen_df.copy()
pred_df["y_true"] = [id2label[int(v)] for v in y_true]
pred_df["y_pred_argmax"] = [id2label[int(v)] for v in y_argmax]
pred_df["y_pred_calibrated"] = [id2label[int(v)] for v in y_calib]
pred_df["p_disinfo"] = p_unseen[:, id_disinfo]
pred_df["p_hate"] = p_unseen[:, id_hate]
pred_df["p_normal"] = p_unseen[:, id_normal]
pred_df["argmax_correct"] = pred_df["y_true"] == pred_df["y_pred_argmax"]
pred_df["calibrated_correct"] = pred_df["y_true"] == pred_df["y_pred_calibrated"]

run_name = RUN_NAME.strip() or datetime.now(timezone.utc).strftime("sbert_%Y%m%d_%H%M%S")
pred_path = REPORT_DIR_RESOLVED / f"{run_name}_unseen_predictions.csv"
summary_path = REPORT_DIR_RESOLVED / f"{run_name}_eval_summary.json"
source_path = REPORT_DIR_RESOLVED / f"{run_name}_source_metrics.csv"
pred_df.to_csv(pred_path, index=False, encoding="utf-8")

source_rows: list[dict[str, str]] = []
for strategy, col in [("argmax", "y_pred_argmax"), ("calibrated", "y_pred_calibrated")]:
    for source, grp in pred_df.groupby("source"):
        yt = grp["y_true"].map(label2id).to_numpy()
        yp = grp[col].map(label2id).to_numpy()
        source_rows.append(
            {
                "run_name": run_name,
                "strategy": strategy,
                "source": source,
                "rows": str(len(grp)),
                "accuracy": f"{accuracy_score(yt, yp):.6f}",
                "macro_f1": f"{f1_score(yt, yp, average='macro', zero_division=0):.6f}",
            }
        )
pd.DataFrame(source_rows).to_csv(source_path, index=False, encoding="utf-8")

summary = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "run_name": run_name,
    "model_root": str(MODEL_ROOT_RESOLVED),
    "unseen_csv": str(UNSEEN_CSV),
    "calibration": {
        "source_train_csv": str(TRAIN_CSV),
        "val_size": VAL_SIZE,
        "seed": SEED,
        "rows": int(len(val_df)),
        "best_thresholds": {"hate": best.t_hate, "disinfo": best.t_disinfo},
        "val_macro_f1": best.val_macro_f1,
        "val_accuracy": best.val_accuracy,
        "val_min_class_f1": best.val_min_class_f1,
    },
    "unseen_metrics": {
        "argmax": m_argmax,
        "calibrated": m_calib,
    },
    "artifacts": {
        "predictions_csv": str(pred_path),
        "source_metrics_csv": str(source_path),
    },
}
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

def leaderboard_row(strategy: str, metrics: dict) -> dict[str, str]:
    rep = metrics["report"]
    return {
        "created_at": summary["created_at"],
        "run_name": run_name,
        "strategy": strategy,
        "model_root": str(MODEL_ROOT_RESOLVED),
        "unseen_csv": str(UNSEEN_CSV),
        "rows": str(len(unseen_df)),
        "accuracy": f"{metrics['accuracy']:.6f}",
        "macro_f1": f"{metrics['macro_f1']:.6f}",
        "f1_disinfo": f"{float(rep['DISINFO']['f1-score']):.6f}",
        "f1_hate": f"{float(rep['HATE']['f1-score']):.6f}",
        "f1_normal": f"{float(rep['NORMAL']['f1-score']):.6f}",
        "recall_disinfo": f"{float(rep['DISINFO']['recall']):.6f}",
        "recall_hate": f"{float(rep['HATE']['recall']):.6f}",
        "recall_normal": f"{float(rep['NORMAL']['recall']):.6f}",
        "precision_disinfo": f"{float(rep['DISINFO']['precision']):.6f}",
        "precision_hate": f"{float(rep['HATE']['precision']):.6f}",
        "precision_normal": f"{float(rep['NORMAL']['precision']):.6f}",
        "threshold_hate": f"{best.t_hate:.2f}",
        "threshold_disinfo": f"{best.t_disinfo:.2f}",
        "summary_json": str(summary_path),
    }

append_leaderboard(LEADERBOARD_CSV, leaderboard_row("argmax", m_argmax))
append_leaderboard(LEADERBOARD_CSV, leaderboard_row("calibrated", m_calib))

log(f"Summary written: {summary_path}")
log(f"Leaderboard updated: {LEADERBOARD_CSV}")
log(f"Unseen macro_f1 | argmax={m_argmax['macro_f1']:.4f} calibrated={m_calib['macro_f1']:.4f}")

summary_path, source_path, LEADERBOARD_CSV
